In [ ]:
!pip install torch torchvision
!pip install pillow 
!pip install imagehash  
!pip install scipy 
!pip install scikit-learn
!pip install transformers

In [3]:
from transformers import AutoImageProcessor, AutoModel
import torch
from PIL import Image
import numpy as np
import time

# Initialize device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model and processor
processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
model = AutoModel.from_pretrained('facebook/dinov2-base').to(device)
model.eval()

def preprocess_images(image_paths):
    """Load and preprocess a list of images."""
    images = []
    valid_paths = []
    
    for path in image_paths:
        try:
            image = Image.open(path).convert('RGB')
            images.append(image)
            valid_paths.append(path)
        except Exception as e:
            print(f"Error loading {path}: {e}")
    
    return images, valid_paths

def extract_features(images):
    """Extract features from a list of images."""
    features = []
    
    for image in images:
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            feature = outputs.last_hidden_state[:, 0, :]  # CLS token
        features.append(feature.cpu().numpy())
    
    return np.array(features)

def compute_similarity_matrix(features):
    """Compute cosine similarity matrix for all image pairs."""
    num_images = features.shape[0]
    similarity_matrix = np.zeros((num_images, num_images))
    
    for i in range(num_images):
        for j in range(num_images):
            similarity = np.dot(features[i], features[j].T) / \
                        (np.linalg.norm(features[i]) * np.linalg.norm(features[j]))
            similarity_matrix[i, j] = similarity
    
    return similarity_matrix

def compute_multiple_image_similarities(image_paths):
    """Compute similarity scores between multiple images."""
    start_time = time.time()
    
    # Load and preprocess images
    print("Loading and preprocessing images...")
    images, valid_paths = preprocess_images(image_paths)
    if not images:
        print("No valid images found.")
        return None
    
    # Extract features
    print("Extracting features...")
    features = extract_features(images)
    features = features.squeeze()  # Remove batch dimension
    
    # Compute similarity matrix
    print("Computing similarity matrix...")
    similarity_matrix = compute_similarity_matrix(features)
    
    # Convert to percentage scores
    percentage_matrix = (similarity_matrix + 1) * 50
    
    # Create results dictionary
    results = {
        "image_paths": valid_paths,
        "similarity_matrix": percentage_matrix.tolist(),
        "processing_time": time.time() - start_time
    }
    
    return results

def print_similarity_matrix(results):
    """Print the similarity matrix in a readable format."""
    if results is None:
        return
    
    paths = results["image_paths"]
    matrix = results["similarity_matrix"]
    
    print("\nSimilarity Matrix (%):")
    print("Image".ljust(40), end="")
    for path in paths:
        print(f"{path.split('/')[-1]:<10}", end="")
    print()
    
    for i, path in enumerate(paths):
        print(f"{path.split('/')[-1]:<40}", end="")
        for j in range(len(paths)):
            print(f"{matrix[i][j]:<10.2f}", end="")
        print()
    
    print(f"\nProcessing time: {results['processing_time']:.2f} seconds")

# Example usage
if __name__ == "__main__":
    image_paths = [
        "/workspace/Notebook/ai/image_similarity_detection_model/mec2_2.jpeg",
        "/workspace/Notebook/ai/image_similarity_detection_model/mec2_1.jpg",
        "/workspace/Notebook/ai/image_similarity_detection_model/car.jpg",
        "/workspace/Notebook/ai/image_similarity_detection_model/cat.jpg"
    ]
    
    results = compute_multiple_image_similarities(image_paths)
    print_similarity_matrix(results)

Loading and preprocessing images...
Extracting features...
Computing similarity matrix...

Similarity Matrix (%):
Image                                   mec2_2.jpegmec2_1.jpgcar.jpg   cat.jpg   
mec2_2.jpeg                             100.00    92.25     50.00     54.62     
mec2_1.jpg                              92.25     100.00    50.55     54.65     
car.jpg                                 50.00     50.55     100.00    50.74     
cat.jpg                                 54.62     54.65     50.74     100.00    

Processing time: 3.16 seconds
